# Notebook 2 of 4: Load into Databricks as Delta tables (Part 4)

Databricks Free Edition cannot reach the shared databases directly, so the clean files from Notebook 1 are uploaded by hand.

**Before running:** run the first code cell below to create the schema and volume. Then, in Catalog, open `workspace`, then `mustafa_shop`, then the `clean_files` volume, click **Upload to this volume**, and upload the 7 Parquet files from the `clean/` folder.

In [ ]:
CATALOG = "workspace"
SCHEMA = "mustafa_shop"   # my own schema, as the brief suggests
DB = f"{CATALOG}.{SCHEMA}"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/clean_files"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.clean_files")

DataFrame[]


### 4.2 Save each file as a Delta table

In [ ]:
for name in ["orders_clean", "weather_daily", "dim_date", "dim_customer", "dim_category"]:
    df = spark.read.parquet(f"{VOLUME_PATH}/{name}.parquet")
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.{name}")
    print(f"{name:15} {df.count():>8,} rows")

orders_clean     200,000 rows
weather_daily      1,986 rows
dim_date             730 rows
dim_customer       2,057 rows
dim_category           6 rows


`fact_orders` starts with the first load only. I drop it first so that when I run this notebook again, version 0 is still the first load and not some old run.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {DB}.fact_orders")

first_load = spark.read.parquet(f"{VOLUME_PATH}/fact_orders_initial.parquet")
first_load.write.format("delta").saveAsTable(f"{DB}.fact_orders")

print(f"Rows after first load: {spark.table(f'{DB}.fact_orders').count():,}")

Rows after first load: 197,153


### 4.3 Second load: orders updated after 2026-10-20
I use MERGE and not append. An order that changed (for example from shipped to delivered) comes through with the same order_id, and append would create a second row for it. MERGE updates the row if the order_id exists and inserts it if not, so running it twice does not create duplicates.

In [ ]:
updates = spark.read.parquet(f"{VOLUME_PATH}/fact_orders_incremental.parquet")
updates.createOrReplaceTempView("order_updates")
print(f"Orders in the second load: {updates.count():,}")

spark.sql(f"""
    MERGE INTO {DB}.fact_orders AS t
    USING order_updates AS u
    ON t.order_id = u.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"Rows after second load: {spark.table(f'{DB}.fact_orders').count():,}")

Orders in the second load: 2,847
Rows after second load: 200,000


### Table history
**Take the screenshot of this output for the submission.**

In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {DB}.fact_orders")
display(history.select("version", "timestamp", "operation", "operationMetrics"))

 version                timestamp              operation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            operationMetrics
       1 2026-11-04T15:37:06.000Z                  MERGE {'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetFilesAdded': '1', 'numTargetBytesAdded': '44660', 'numTargetBytesRemoved': '0', 'numTargetDeletionVect

### 4.4 Time travel: version 0 against the current version

In [ ]:
compare = spark.sql(f"""
    SELECT 'version 0' AS version, COUNT(*) AS row_count FROM {DB}.fact_orders VERSION AS OF 0
    UNION ALL
    SELECT 'current', COUNT(*) FROM {DB}.fact_orders
""")
display(compare)

rows = {r["version"]: r["row_count"] for r in compare.collect()}
print(f"The second load added {rows['current'] - rows['version 0']:,} rows.")

# Checks that nothing was duplicated
fact = spark.table(f"{DB}.fact_orders")
print("Duplicate order_ids:", fact.count() - fact.select("order_id").distinct().count())

  version  row_count
version 0     197153
  current     200000
The second load added 2,847 rows.
Duplicate order_ids: 0
